# Imports and setup

In [21]:
# Necessary imports and setup
import sys
import os

# Add execution tracking to debug duplicate output
print("=== STARTING IMPORTS CELL ===")

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

import multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Configure JAX GPU memory settings BEFORE importing jax - OPTIMIZED FOR 40GB A100
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.99'  # Use 98% of GPU memory (~39.2GB out of 40GB)
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'cuda_async'  # Use platform allocator
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'  # Don't preallocate - grow as needed to avoid fragmentation
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # Allow dynamic growth

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

import jax
from jax import numpy as jp
from jax.lib import xla_bridge

print("Device count: ", jax.device_count())

# Configure JAX to use only GPU1

print(f"CUDA_VISIBLE_DEVICES set to: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"JAX memory fraction set to: {os.environ.get('XLA_PYTHON_CLIENT_MEM_FRACTION')}")
print(f"JAX preallocate disabled: {os.environ.get('XLA_PYTHON_CLIENT_PREALLOCATE')}")

print("JAX backend info:")
print(f"Platform: {xla_bridge.get_backend().platform}")
print(f"Device count: {xla_bridge.get_backend().device_count()}")
print(f"Devices: {xla_bridge.get_backend().devices()}")

# JAX configuration optimized for large workloads
jax.config.update('jax_enable_x64', False)  # Use float32 to save memory
jax.config.update('jax_traceback_filtering', 'off')
# Use bfloat16 for even better memory efficiency (optional - comment out if you need float32 precision)
# jax.config.update('jax_default_matmul_precision', 'bfloat16')

# Check GPU availability and memory
gpu_available = jax.devices()[0].platform == 'gpu'
print(f"GPU available: {gpu_available}")

if gpu_available:
    gpu_device = jax.devices('gpu')[0]
    print(f"GPU device: {gpu_device}")
else:
    print("No GPU device found.")

import signal
import json
import functools
import mujoco
from datetime import datetime
from pathlib import Path
import imageio
import gc

print("Basic imports completed...")

# Brax and training imports
from brax.io import model
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from flax.training import orbax_utils
from orbax import checkpoint as ocp
from mujoco_playground.config import locomotion_params
from mujoco_playground import wrapper
from tensorboardX import SummaryWriter

print("Brax imports completed...")

# Task-specific imports
from tasks.common.randomize import domain_randomize as reachbot_randomize
from utils.telegram_messenger import send_message_sync

print("Task-specific imports completed...")

# Global variables
ENV_STR = 'Go1Getup'
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


from tasks.getup.getup import Getup as ReachbotGetup
from tasks.common.randomize import domain_randomize as reachbot_randomize
from mujoco_playground import registry
from utils.utils import JaxArrayEncoder, convert_to_dict

print("=== ALL IMPORTS LOADED SUCCESSFULLY! ===")

=== STARTING IMPORTS CELL ===
Device count:  1
CUDA_VISIBLE_DEVICES set to: 0
JAX memory fraction set to: 0.99
JAX preallocate disabled: false
JAX backend info:
Platform: gpu
Device count: 1
Devices: [CudaDevice(id=0)]
GPU available: True
GPU device: cuda:0
Basic imports completed...
Brax imports completed...
Task-specific imports completed...
=== ALL IMPORTS LOADED SUCCESSFULLY! ===


# Environment config

In [22]:
from tasks.getup.getup import default_config as reachbot_config
env_cfg = reachbot_config()

print("Environment configuration completed!")


Environment configuration completed!


# Training parameters

In [23]:
ppo_params = locomotion_params.brax_ppo_config(ENV_STR)
ppo_training_params = dict(ppo_params)
ppo_training_params["num_evals"] = 20

ppo_training_params["num_timesteps"] = 30_000_000

action_size = 12 # Action space size for Reachbot (Basic without grippers)

# Training


In [24]:
import os
import threading
print(f"🔥 PID: {os.getpid()} | Thread: {threading.current_thread().ident} | Time: {datetime.now()}")

# Training execution
# Create log directory for training run
datetime_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# Use main directory logs folder instead of task-specific logs
main_dir = os.path.dirname(os.path.dirname(os.getcwd()))
logdir = os.path.join(main_dir, "logs", "getup-"+datetime_str)
os.makedirs(logdir, exist_ok=True)


# Create environment
env = ReachbotGetup(config=env_cfg)

# Initialize tracking variables
timesteps = []
rewards = []
total_rewards = []
total_rewards_std = []
times = [datetime.now()]

writer = SummaryWriter(logdir=logdir)

# Progress tracking function
def progress(num_steps, metrics):
    """Function to track progress and log metrics during training."""
    print(f"Progress at step {num_steps}: {metrics}")
    # Log to TensorBoard
    for key, value in metrics.items():
        if not (jp.isnan(value) or jp.isinf(value)):
            writer.add_scalar(key, value, num_steps)
        else:
            print(f"Warning: Skipping NaN/Inf value for metric '{key}' at step {num_steps}")

    if "eval/episode_reward" in metrics:
        episode_reward = metrics["eval/episode_reward"]
        if jp.isnan(episode_reward) or jp.isinf(episode_reward):
            print("Warning: NaN/Inf reward encountered, aborting.")
            run_duration = str(datetime.now() - times[0])
            send_message_sync(
                task="Getup RL Training",
                duration=run_duration,
                result="Failed: NaN/Inf reward encountered"
            )
            raise ValueError(f"NaN/Inf reward encountered at step {num_steps}: {episode_reward}")
        
        times.append(datetime.now())
        timesteps.append(num_steps)
        total_rewards.append(episode_reward)
        total_rewards_std.append(metrics["eval/episode_reward_std"])
    
        writer.flush()
        metrics["timesteps"] = num_steps
        metrics["time"] = (times[-1] - times[0]).total_seconds()
        rewards.append(metrics)
        
        percent_complete = (num_steps / ppo_training_params["num_timesteps"]) * 100
        if num_steps == 0:
            remaining_time_str = "unknown"
        else:
            elapsed_time = (times[-1] - times[0]).total_seconds()
            remaining_steps = ppo_training_params["num_timesteps"] - num_steps
            remaining_time = remaining_steps * elapsed_time / num_steps / 60
            remaining_time_str = f"{remaining_time:.2f}"
        
        print(f"step: {num_steps}/{ppo_training_params['num_timesteps']} ({percent_complete:.1f}%), reward: {total_rewards[-1]:.3f} +/- {total_rewards_std[-1]:.3f}, time passed (min): {(times[-1] - times[0]).total_seconds() / 60:.2f} min, calculated time left (min): {remaining_time_str} min")

# Network factory setup



network_factory = ppo_networks.make_ppo_networks(observation_size=env.observation_size, action_size=env.action_size)
    
ppo_params = locomotion_params.brax_ppo_config(ENV_STR)
if "network_factory" in ppo_params:
    network_factory = functools.partial(
        ppo_networks.make_ppo_networks,
        **ppo_params.network_factory
    )
    del ppo_training_params["network_factory"]

# Checkpoint saving function
def policy_params_fn(current_step, make_policy, params):
    del make_policy  # Unused.
    orbax_checkpointer = ocp.PyTreeCheckpointer()
    save_args = orbax_utils.save_args_from_target(params)
    checkpoint_path = os.path.join(logdir, 'checkpoints')
    path = os.path.join(checkpoint_path, f"{current_step}")
    abs_path = os.path.abspath(path)
    orbax_checkpointer.save(abs_path, params, force=True, save_args=save_args)

# Save configurations
print("Saving configs")
# Build raw configs with possibly non-serializable objects
configs_raw = {
    "env_cfg": env_cfg,
    "ppo_params": ppo_training_params,
}
# Convert ConfigDicts, JAX/NumPy scalars/arrays, etc. into JSON-safe python types
configs = convert_to_dict(configs_raw)

def replace_infinity(obj):
    import math
    if isinstance(obj, dict):
        return {k: replace_infinity(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [replace_infinity(v) for v in obj]
    elif isinstance(obj, float):
        if math.isinf(obj):
            return 1e308 if obj > 0 else -1e308
        if math.isnan(obj):
            return 0.0
    return obj

configs = replace_infinity(configs)
config_path = os.path.join(logdir, 'config.json')
with open(config_path, "w", encoding="utf-8") as fp:
    json.dump(configs, fp, indent=4, cls=JaxArrayEncoder)
print(f"Configuration saved to {config_path}")
writer.add_text('config', json.dumps(configs, indent=4, cls=JaxArrayEncoder))

# DISABLE DOMAIN RANDOMIZATION TO FIX JAX TRACER LEAK ERROR
# The registry.get_domain_randomizer("Go1Getup") was causing a JAX tracer leak
# randomizer = registry.get_domain_randomizer("Go1Getup")
randomizer = None  # No domain randomization for now

train_fn = functools.partial(
    ppo.train, 
    **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    policy_params_fn=policy_params_fn,
    max_devices_per_host=1,
    randomization_fn=randomizer,  # This will be None, disabling randomization
    log_training_metrics=False,
)

# Run training
print("Training the model...")
try:
    make_inference_fn, params, metrics = train_fn(
        environment=env,
        wrap_env_fn=wrapper.wrap_for_brax_training,
    )
    print("Training completed successfully!")
except Exception as e:
    import traceback
    run_duration = str(datetime.now() - times[0])
    send_message_sync(
        task="Getup RL Training",
        duration=run_duration,
        result=f"Failed: {e}"
    )
    traceback.print_exc()
    raise

print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

# Save results
results_path = os.path.join(logdir, 'results.txt')
with open(results_path, 'w') as f:
    for i in range(len(total_rewards)):
        f.write(f"step: {timesteps[i]}, reward: {total_rewards[i]}, reward_std: {total_rewards_std[i]}\n")
    f.write(f"Time to jit: {times[1] - times[0]}\n")
    f.write(f"Time to train: {times[-1] - times[1]}\n")

# Save rewards as JSON
def nest_flat_dict(flat_dict):
    nested_dict = {}
    for key, value in flat_dict.items():
        parts = key.split('/')
        d = nested_dict
        for i, part in enumerate(parts):
            is_last_part = (i == len(parts) - 1)
            if is_last_part:
                if isinstance(d.get(part), dict):
                    d[part]['value'] = value
                else:
                    d[part] = value
            else:
                if not isinstance(d.get(part), dict):
                    d[part] = {'value': d[part]} if part in d else {}
                d = d[part]
    return nested_dict

nested_rewards = [nest_flat_dict(r) for r in rewards]
rewards_path = os.path.join(logdir, 'rewards.json')
with open(rewards_path, 'w') as fp:
    json.dump(nested_rewards, fp, indent=4, cls=JaxArrayEncoder)

# Save trained parameters
params_path = os.path.join(logdir, 'params')
model.save_params(params_path, params)

print(f"Training completed! Results saved to: {logdir}")
print(f"Final reward: {total_rewards[-1]:.3f} ± {total_rewards_std[-1]:.3f}")

# Store these variables for the video generation cell
trained_params = params
trained_make_inference_fn = make_inference_fn
trained_env = env
trained_logdir = logdir
# Video creation from trained model
# Free up training memory before rendering
del train_fn, network_factory, writer
gc.collect()

# Use the already loaded model and environment from the training cell
env = trained_env
params = trained_params
make_inference_fn = trained_make_inference_fn
logdir = trained_logdir

# Setup JIT compiled functions for inference
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
inference_fn = make_inference_fn(params, deterministic=True)
jit_inference_fn = jax.jit(inference_fn)

print("Setting up rollout for video creation...")

# Rollout parameters
rng = jax.random.PRNGKey(0)
rollout = []
n_episodes = 1
rollout_steps = 1200

# Set command (if needed for environment)
x_vel = 0.2
y_vel = 0.2
yaw_vel = 0.0
command = jp.array([x_vel, y_vel, yaw_vel])

# Rollout policy and record simulation
print(f"Running rollout for {n_episodes} episode(s) with {rollout_steps} steps each...")
for episode in range(n_episodes):
    print(f"Episode {episode + 1}/{n_episodes}")
    state = jit_reset(rng)
    rollout.append(state)
    episode_reward = 0.0
    
    for i in range(rollout_steps):
        if i % 500 == 0:
            print(f"  Step {i}/{rollout_steps} - Total Reward: {episode_reward:.3f}")
            
        act_rng, rng = jax.random.split(rng)
        ctrl, _ = jit_inference_fn(state.obs, act_rng)
        
        # Check for numerical issues
        if jp.any(jp.isinf(ctrl)) or jp.any(jp.isnan(ctrl)):
            print(f"Numerical issue detected in control at step {i}. Stopping rollout.")
            break
            
        state = jit_step(state, ctrl)
        
        # Set command if the environment supports it
        if hasattr(state, 'info') and 'command' in state.info:
            state.info["command"] = command

        episode_reward += state.reward
            
        rollout.append(state)

    print(f"Rollout completed with {len(rollout)} states")

    # Render video
    print("Rendering video...")
    render_every = 1  # Render every frame
    width = 1920      # Full HD width
    height = 1080     # Full HD height

    frames = env.render(rollout[::render_every], camera='track_global', width=width, height=height)
    print(f"Rendered {len(frames)} frames")

    # Save video
    video_path = os.path.join(logdir, f'posttraining_{episode_reward:.2f}.mp4')
    fps = 1.0 / env.dt

    print(f"Saving video to {video_path} at {fps} FPS...")
    imageio.mimsave(video_path, frames, fps=fps)
    print(f"Video saved successfully to {video_path}")

# Send completion notification
run_duration = str(times[-1] - times[0])
if total_rewards:
    result = f"Final reward: {total_rewards[-1]:.3f} ± {total_rewards_std[-1]:.3f}"
else:
    result = "No rewards recorded."

send_message_sync(
    task="Getup RL Training",
    duration=run_duration,
    result=result
)

print("\n=== TRAINING AND VIDEO CREATION COMPLETE ===")
print(f"Log directory: {logdir}")
print(f"Video file: {video_path}")
print(f"Training duration: {run_duration}")
print(f"Final result: {result}")

🔥 PID: 2590727 | Thread: 140037234688640 | Time: 2025-10-09 22:24:38.589168
Precomputed 100 LIDAR ray directions
CaveExplore task action space: 12
Saving configs
Configuration saved to /home/ga53voq/master_thesis/logs/getup-2025-10-09_22-24-38/config.json
Training the model...


2025-10-09 22:27:00.411812: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-10-09 22:27:00.411924: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-10-09 22:27:00.411938: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Progress at step 0: {'eval/walltime': 107.18355870246887, 'eval/episode_reward': Array(2.0624795, dtype=float32), 'eval/episode_reward/action_rate': Array(0., dtype=float32), 'eval/episode_reward/dof_acc': Array(0., dtype=float32), 'eval/episode_reward/dof_pos_limits': Array(0., dtype=float32), 'eval/episode_reward/dof_vel': Array(0., dtype=float32), 'eval/episode_reward/orientation': Array(15.652567, dtype=float32), 'eval/episode_reward/posture': Array(2.844337, dtype=float32), 'eval/episode_reward/shoulder_torque': Array(-0.8300524, dtype=float32), 'eval/episode_reward/stand_still': Array(1.0650988, dtype=float32), 'eval/episode_reward/torques': Array(0., dtype=float32), 'eval/episode_reward/torso_height': Array(83.81491, dtype=float32), 'eval/episode_reward/vertical_velocity': Array(0., dtype=float32), 'eval/episode_reward_std': Array(0.8595763, dtype=float32), 'eval/episode_reward/action_rate_std': Array(0., dtype=float32), 'eval/episode_reward/dof_acc_std': Array(0., dtype=float32

2025-10-09 22:28:39.184630: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-10-09 22:28:39.186019: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Progress at step 1638400: {'eval/walltime': 110.84893941879272, 'training/sps': np.float64(14907.803759292534), 'training/walltime': 109.90217113494873, 'training/entropy_loss': Array(-0.07359059, dtype=float32), 'training/policy_loss': Array(0.00936505, dtype=float32), 'training/total_loss': Array(-0.0639491, dtype=float32), 'training/v_loss': Array(0.00027644, dtype=float32), 'eval/episode_reward': Array(3.254243, dtype=float32), 'eval/episode_reward/action_rate': Array(0., dtype=float32), 'eval/episode_reward/dof_acc': Array(0., dtype=float32), 'eval/episode_reward/dof_pos_limits': Array(0., dtype=float32), 'eval/episode_reward/dof_vel': Array(0., dtype=float32), 'eval/episode_reward/orientation': Array(43.478565, dtype=float32), 'eval/episode_reward/posture': Array(4.098839, dtype=float32), 'eval/episode_reward/shoulder_torque': Array(-0.97426766, dtype=float32), 'eval/episode_reward/stand_still': Array(1.5330203, dtype=float32), 'eval/episode_reward/torques': Array(0., dtype=float

2025-10-09 22:35:07.976371: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


  Step 0/1200 - Total Reward: 0.000
  Step 500/1200 - Total Reward: 75.713
  Step 1000/1200 - Total Reward: 163.652
Rollout completed with 1201 states
Rendering video...


100%|██████████| 1201/1201 [00:30<00:00, 39.92it/s]
/home/ga53voq/.conda/envs/pyenv/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(


Rendered 1201 frames
Saving video to /home/ga53voq/master_thesis/logs/getup-2025-10-09_22-24-38/posttraining_198.65.mp4 at 50.0 FPS...
Video saved successfully to /home/ga53voq/master_thesis/logs/getup-2025-10-09_22-24-38/posttraining_198.65.mp4

=== TRAINING AND VIDEO CREATION COMPLETE ===
Log directory: /home/ga53voq/master_thesis/logs/getup-2025-10-09_22-24-38
Video file: /home/ga53voq/master_thesis/logs/getup-2025-10-09_22-24-38/posttraining_198.65.mp4
Training duration: 0:10:09.992862
Final result: Final reward: 47.324 ± 6.272


# Output video render

In [25]:
# Video creation from trained model


# Use the already loaded model and environment from the training cell
env = trained_env
params = trained_params
make_inference_fn = trained_make_inference_fn
logdir = trained_logdir

# Setup JIT compiled functions for inference
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
inference_fn = make_inference_fn(params, deterministic=False)
jit_inference_fn = jax.jit(inference_fn)

print("Setting up rollout for video creation...")

# Rollout parameters
rng = jax.random.PRNGKey(0)
rollout = []
n_episodes = 1
rollout_steps = ppo_training_params.get("episode_length", 1200)  # Default to 1200 if not specified

# Set command (if needed for environment)
x_vel = 0.2
y_vel = 0.2
yaw_vel = 0.0
command = jp.array([x_vel, y_vel, yaw_vel])

# Rollout policy and record simulation
print(f"Running rollout for {n_episodes} episode(s) with {rollout_steps} steps each...")
episode_rewards = []

for episode in range(n_episodes):
    print(f"Episode {episode + 1}/{n_episodes}")
    state = jit_reset(rng)
    rollout = [state]  # Reset rollout for each episode
    episode_reward = 0.0
    
    for i in range(rollout_steps):
        if i % 500 == 0:
            print(f"  Step {i}/{rollout_steps}, Current reward: {episode_reward:.3f}")
            
        act_rng, rng = jax.random.split(rng)
        ctrl, _ = jit_inference_fn(state.obs, act_rng)
        
        # Check for numerical issues
        if jp.any(jp.isinf(ctrl)) or jp.any(jp.isnan(ctrl)):
            print(f"Numerical issue detected in control at step {i}. Stopping rollout.")
            break
            
        state = jit_step(state, ctrl)
        
        # Accumulate reward for this episode
        episode_reward += float(state.reward)
        
        if state.done:
            print(f"Episode {episode + 1} ended at step {i} with reward: {episode_reward:.3f}")
            break
            
        rollout.append(state)

    episode_rewards.append(episode_reward)
    print(f"Episode {episode + 1} completed with {len(rollout)} states and total reward: {episode_reward:.3f}")

    # Render video
    print("Rendering video...")
    render_every = 1  # Render every frame
    width = 1920      # Full HD width
    height = 1080     # Full HD height

    frames = env.render(rollout[::render_every], camera='track_global', width=width, height=height)
    print(f"Rendered {len(frames)} frames")

    # Save video
    video_path = os.path.join(logdir, f'posttraining_episode_{episode}_reward_{episode_reward:.1f}.mp4')
    fps = 1.0 / env.dt

    print(f"Saving video to {video_path} at {fps} FPS...")
    imageio.mimsave(video_path, frames, fps=fps)
    print(f"Video saved successfully: Episode {episode + 1}, Reward: {episode_reward:.3f}")

# Print summary of all episodes
print("\n=== EPISODE REWARD SUMMARY ===")
for i, reward in enumerate(episode_rewards):
    print(f"Episode {i + 1}: {reward:.3f}")
print(f"Average reward: {sum(episode_rewards)/len(episode_rewards):.3f}")
print(f"Best episode: {episode_rewards.index(max(episode_rewards)) + 1} with reward {max(episode_rewards):.3f}")
print(f"Worst episode: {episode_rewards.index(min(episode_rewards)) + 1} with reward {min(episode_rewards):.3f}")

# Send completion notification
run_duration = str(times[-1] - times[0])
if total_rewards:
    training_result = f"Training final reward: {total_rewards[-1]:.3f} ± {total_rewards_std[-1]:.3f}"
else:
    training_result = "No training rewards recorded."

# Include episode rewards in notification
episode_summary = f"Episode rewards: {[f'{r:.1f}' for r in episode_rewards]}, Avg: {sum(episode_rewards)/len(episode_rewards):.1f}"

send_message_sync(
    task="Getup RL Training",
    duration=run_duration,
    result=f"{training_result}\n{episode_summary}"
)

print("\n=== TRAINING AND VIDEO CREATION COMPLETE ===")
print(f"Log directory: {logdir}")
print(f"Training duration: {run_duration}")
print(f"Training result: {training_result}")
print(f"Episode summary: {episode_summary}")

Setting up rollout for video creation...
Running rollout for 1 episode(s) with 300 steps each...
Episode 1/1
  Step 0/300, Current reward: 0.000
Episode 1 completed with 301 states and total reward: 44.037
Rendering video...


100%|██████████| 301/301 [00:08<00:00, 35.38it/s]


Rendered 301 frames
Saving video to /home/ga53voq/master_thesis/logs/getup-2025-10-09_22-24-38/posttraining_episode_0_reward_44.0.mp4 at 50.0 FPS...
Video saved successfully: Episode 1, Reward: 44.037

=== EPISODE REWARD SUMMARY ===
Episode 1: 44.037
Average reward: 44.037
Best episode: 1 with reward 44.037
Worst episode: 1 with reward 44.037

=== TRAINING AND VIDEO CREATION COMPLETE ===
Log directory: /home/ga53voq/master_thesis/logs/getup-2025-10-09_22-24-38
Training duration: 0:10:09.992862
Training result: Training final reward: 47.324 ± 6.272
Episode summary: Episode rewards: ['44.0'], Avg: 44.0
